In [1]:
import duckdb
con = duckdb.connect("../data/processed/retail.duckdb", read_only=True)

The suppliers

In [2]:
con.sql("SELECT * FROM sim.supplier ORDER BY supplier_id")

┌─────────────────┬───────────┬──────────────────┬───────────┬──────────────┐
│   supplier_id   │ category  │ quoted_lead_days │ late_rate │ partial_rate │
│     varchar     │  varchar  │      int32       │  double   │    double    │
├─────────────────┼───────────┼──────────────────┼───────────┼──────────────┤
│ SUP_FOODS_1     │ FOODS     │                2 │     0.089 │        0.048 │
│ SUP_FOODS_2     │ FOODS     │                2 │     0.093 │        0.058 │
│ SUP_FOODS_3     │ FOODS     │                2 │     0.055 │        0.069 │
│ SUP_FOODS_4     │ FOODS     │                2 │     0.088 │        0.061 │
│ SUP_HOBBIES_1   │ HOBBIES   │                7 │     0.194 │        0.039 │
│ SUP_HOBBIES_2   │ HOBBIES   │                7 │     0.205 │        0.033 │
│ SUP_HOBBIES_3   │ HOBBIES   │                7 │     0.233 │        0.055 │
│ SUP_HOUSEHOLD_1 │ HOUSEHOLD │                4 │     0.106 │        0.048 │
│ SUP_HOUSEHOLD_2 │ HOUSEHOLD │                4 │     0.119 │  

How many products does each supplier handle

In [3]:
con.sql("""
    SELECT supplier_id, COUNT(*) AS items
    FROM sim.item_supply
    GROUP BY supplier_id
    ORDER BY supplier_id
""")

┌─────────────────┬───────┐
│   supplier_id   │ items │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ SUP_FOODS_1     │   350 │
│ SUP_FOODS_2     │   353 │
│ SUP_FOODS_3     │   364 │
│ SUP_FOODS_4     │   370 │
│ SUP_HOBBIES_1   │   178 │
│ SUP_HOBBIES_2   │   196 │
│ SUP_HOBBIES_3   │   191 │
│ SUP_HOUSEHOLD_1 │   375 │
│ SUP_HOUSEHOLD_2 │   344 │
│ SUP_HOUSEHOLD_3 │   328 │
└─────────────────┴───────┘
  10 rows       2 columns

The ordering terms per category

In [4]:
con.sql("""
    SELECT cat_id,
           COUNT(*)                AS items,
           MIN(pack_size)          AS pack_size,
           MIN(review_days)        AS review_days,
           MIN(service_level)      AS service_level,
           ROUND(MIN(z), 3)        AS z
    FROM sim.item_supply
    GROUP BY cat_id
    ORDER BY cat_id
""")

┌───────────┬───────┬───────────┬─────────────┬───────────────┬────────┐
│  cat_id   │ items │ pack_size │ review_days │ service_level │   z    │
│  varchar  │ int64 │   int32   │    int32    │    double     │ double │
├───────────┼───────┼───────────┼─────────────┼───────────────┼────────┤
│ FOODS     │  1437 │         6 │           1 │          0.95 │  1.645 │
│ HOBBIES   │   565 │         4 │           7 │           0.9 │  1.282 │
│ HOUSEHOLD │  1047 │        12 │           3 │          0.92 │  1.405 │
└───────────┴───────┴───────────┴─────────────┴───────────────┴────────┘

Does every product in the sales data have a supplier? 

In [5]:
con.sql("""
    SELECT COUNT(*) AS products_without_supplier
    FROM (SELECT DISTINCT item_id FROM raw.sales) s
    LEFT JOIN sim.item_supply i USING (item_id)
    WHERE i.item_id IS NULL
""")

┌───────────────────────────┐
│ products_without_supplier │
│           int64           │
├───────────────────────────┤
│                         0 │
└───────────────────────────┘

How much volume does each supplier handle?

In [6]:
con.sql("""
    SELECT i.supplier_id, SUM(s.units) AS units_sold
    FROM raw.sales s
    JOIN sim.item_supply i USING (item_id)
    GROUP BY i.supplier_id
    ORDER BY units_sold DESC
""")

┌─────────────────┬────────────┐
│   supplier_id   │ units_sold │
│     varchar     │   int128   │
├─────────────────┼────────────┤
│ SUP_FOODS_3     │    3440451 │
│ SUP_FOODS_2     │    3199862 │
│ SUP_FOODS_4     │    3161452 │
│ SUP_FOODS_1     │    3027735 │
│ SUP_HOUSEHOLD_1 │    1431288 │
│ SUP_HOUSEHOLD_2 │    1277308 │
│ SUP_HOUSEHOLD_3 │    1251260 │
│ SUP_HOBBIES_2   │     705707 │
│ SUP_HOBBIES_1   │     671309 │
│ SUP_HOBBIES_3   │     620205 │
└─────────────────┴────────────┘
  10 rows            2 columns

In [7]:
con.close()